# mAP Diagnostic
Finds the root cause of low mAP by checking GT ↔ image name alignment.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/datasets/jeffreyamc/oxford-paris-buildings-v2')
GT_DIR       = Path('/kaggle/working/retrieval_output/ground_truth')

def find_gt_dir(gt_root):
    files = list(gt_root.rglob('*_query.txt'))
    return files[0].parent if files else None

gt_flat = find_gt_dir(GT_DIR)
print(f'GT directory: {gt_flat}')

## Check 1 — Do GT stems match actual image filenames?

In [ ]:
# Collect all image stems from the Kaggle dataset
all_stems = set()
for ds in ['oxford', 'paris']:
    ds_root = DATASET_ROOT / ds
    if not ds_root.exists(): continue
    for p in ds_root.rglob('*.jpg'):
        all_stems.add(p.stem)

print(f'Total image stems in dataset: {len(all_stems)}')
# Show a sample
sample = sorted(all_stems)[:10]
print(f'\nSample image stems:')
for s in sample:
    print(f'  {s}')

In [ ]:
# Parse ALL GT files and collect every stem they reference
def clean(s):
    for pfx in ('oxc1_', 'paris_', 'oxf_'):
        s = s.replace(pfx, '')
    return s.strip()

gt_stems_query    = set()   # stems referenced as query images
gt_stems_positive = set()   # stems in good/ok files
gt_stems_junk     = set()   # stems in junk files

for qf in sorted(gt_flat.glob('*_query.txt')):
    with open(qf) as f:
        parts = f.read().strip().split()
    gt_stems_query.add(clean(parts[0]))

for suffix in ('good', 'ok'):
    for f in gt_flat.glob(f'*_{suffix}.txt'):
        with open(f) as fh:
            for line in fh:
                if line.strip():
                    gt_stems_positive.add(clean(line.strip()))

for f in gt_flat.glob('*_junk.txt'):
    with open(f) as fh:
        for line in fh:
            if line.strip():
                gt_stems_junk.add(clean(line.strip()))

all_gt_stems = gt_stems_query | gt_stems_positive | gt_stems_junk

print(f'GT references {len(all_gt_stems)} unique image stems')
print(f'  Query stems    : {len(gt_stems_query)}')
print(f'  Positive stems : {len(gt_stems_positive)}')
print(f'  Junk stems     : {len(gt_stems_junk)}')
print(f'\nSample GT stems (after cleaning prefix):')
for s in sorted(gt_stems_query)[:5]:
    print(f'  {s}')

In [ ]:
# KEY CHECK: How many GT stems exist in the actual image files?
found_query    = gt_stems_query    & all_stems
found_positive = gt_stems_positive & all_stems
found_junk     = gt_stems_junk     & all_stems

missing_query    = gt_stems_query    - all_stems
missing_positive = gt_stems_positive - all_stems

print('='*55)
print('GT ↔ IMAGE ALIGNMENT CHECK')
print('='*55)
print(f'Query images  found : {len(found_query):4d} / {len(gt_stems_query):4d}'
      f'  ({100*len(found_query)/max(len(gt_stems_query),1):.1f}%)')
print(f'Positive imgs found : {len(found_positive):4d} / {len(gt_stems_positive):4d}'
      f'  ({100*len(found_positive)/max(len(gt_stems_positive),1):.1f}%)')
print(f'Junk imgs     found : {len(found_junk):4d} / {len(gt_stems_junk):4d}'
      f'  ({100*len(found_junk)/max(len(gt_stems_junk),1):.1f}%)')

print(f'\nMissing QUERY stems  : {len(missing_query)}')
if missing_query:
    for s in sorted(missing_query)[:5]:
        print(f'  {s}')

print(f'\nMissing POSITIVE stems : {len(missing_positive)}')
if missing_positive:
    for s in sorted(missing_positive)[:5]:
        print(f'  {s}')

## Check 2 — Manual AP calculation on one query

In [ ]:
# Pick first query and inspect manually
qf = sorted(gt_flat.glob('*_query.txt'))[0]
q_stem = qf.stem.replace('_query', '')

with open(qf) as f: parts = f.read().strip().split()
q_img = clean(parts[0])

def read_set(gt_flat, stem, sfx):
    p = gt_flat / f'{stem}_{sfx}.txt'
    if not p.exists(): return set()
    with open(p) as f: return {clean(l) for l in f if l.strip()}

good = read_set(gt_flat, q_stem, 'good')
ok   = read_set(gt_flat, q_stem, 'ok')
junk = read_set(gt_flat, q_stem, 'junk')

print(f'Query: {q_stem}')
print(f'  query_img : "{q_img}"  →  in dataset: {q_img in all_stems}')
print(f'  #good     : {len(good):3d}  →  found in dataset: {len(good & all_stems)}')
print(f'  #ok       : {len(ok):3d}  →  found in dataset: {len(ok & all_stems)}')
print(f'  #junk     : {len(junk):3d}  →  found in dataset: {len(junk & all_stems)}')

print(f'\nSample good stems from GT:')
for s in sorted(good)[:5]:
    print(f'  GT:  "{s}"   in_dataset={s in all_stems}')

# Also show actual image stems from the same landmark folder
landmark = q_stem.rsplit('_', 1)[0]   # e.g. 'all_souls_1' → 'all_souls'
lm_path  = DATASET_ROOT / 'oxford' / landmark
if not lm_path.exists():
    lm_path = DATASET_ROOT / 'paris' / landmark
if lm_path.exists():
    actual_stems = [p.stem for p in lm_path.glob('*.jpg')][:5]
    print(f'\nActual image stems in {landmark}/:')
    for s in sorted(actual_stems):
        print(f'  IMG: "{s}"')

## Check 3 — Verify one query's retrieval score directly

In [ ]:
# Load the saved DB index and check if a known positive
# actually gets a high similarity score
import numpy as np, pickle
CACHE_DIR = Path('/kaggle/working/retrieval_output/cache')

db_index_files = list(CACHE_DIR.glob('db_index_*.npy'))
stems_file     = CACHE_DIR / 'db_stems.pkl'

if not db_index_files or not stems_file.exists():
    print('DB index not found — run the main pipeline first')
else:
    DB_VLAD = np.load(db_index_files[0])
    with open(stems_file, 'rb') as f:
        db_stems = pickle.load(f)
    stem_to_idx = {s: i for i, s in enumerate(db_stems)}

    print(f'DB index: {DB_VLAD.shape}')
    print(f'Sample DB stems: {db_stems[:5]}')

    # Check if query image is in DB
    q_idx = stem_to_idx.get(q_img)
    print(f'\nQuery image "{q_img}" in DB: {q_idx is not None}')

    if good & set(db_stems):
        # Pick a known positive and check its similarity to the query
        known_pos = next(iter(good & set(db_stems)))
        pos_idx   = stem_to_idx[known_pos]
        q_idx_db  = stem_to_idx.get(q_img)

        if q_idx_db is not None:
            sim = float(DB_VLAD[q_idx_db] @ DB_VLAD[pos_idx])
            # Random baseline
            rand_sims = DB_VLAD[q_idx_db] @ DB_VLAD[np.random.choice(len(DB_VLAD), 100)].T
            print(f'\nSimilarity: query ↔ known_positive "{known_pos}" = {sim:.4f}')
            print(f'Random baseline (mean of 100 random):              = {rand_sims.mean():.4f}')
            print(f'\n→ If these are similar, the embeddings are not discriminative enough')
            print(f'→ If sim(positive) >> random, the issue is in evaluation (name mismatch)')
        else:
            print(f'Query image not in DB — cannot compute similarity')
    else:
        print(f'\n⚠  NO positive images from GT found in DB — this is the problem!')
        print(f'   GT good stems:  {sorted(good)[:3]}')
        print(f'   DB stems sample:{db_stems[:3]}')